# Multi-Step Intelligent Advisor System

## Scenario
You are building an **AI-powered advisor system** for students and early-career professionals.  
The system should not jump to one answer directly. Instead, it should work in **multiple steps**:

1. Understand the user's goal  
2. Analyze current skills and constraints  
3. Generate multiple strategy options  
4. Prioritize the best path  
5. Return a practical action plan with risks and next steps  

## What this notebook includes
- **Groq API integration** for LLM reasoning
- **FastAPI backend** so the logic can behave like a real API system
- **Multi-step pipeline** instead of a single-shot prompt
- **Gradio UI** so it runs nicely in Google Colab
- **Heavy comments** so the flow is easy to understand during viva / demo / review

## Suggested use case
This can be positioned as:
- Career advisor
- Learning roadmap advisor
- Project planning advisor
- Skill-gap analysis assistant

In [ ]:
# Install required packages.
# groq   -> LLM API client
# fastapi / uvicorn -> API layer
# gradio -> interactive UI for Colab
# pandas -> optional display / formatting support
!pip install -q groq fastapi uvicorn gradio pandas nest_asyncio

In [ ]:
# ============================================================
# IMPORTS + API KEY SETUP
# ============================================================

import os
import json
import re
from typing import Dict, Any, List

import pandas as pd
import gradio as gr
import nest_asyncio

from fastapi import FastAPI
from pydantic import BaseModel, Field
from fastapi.testclient import TestClient

from groq import Groq

# This helps when running async-related tools inside Colab notebooks.
nest_asyncio.apply()

# ------------------------------------------------------------
# Load the API key safely.
# In Colab:
#   1. Open the left panel -> Secrets
#   2. Add GROQ_API_KEY
#   3. Re-run this cell
# ------------------------------------------------------------
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    try:
        from google.colab import userdata
        GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    except Exception:
        GROQ_API_KEY = None

client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

# Current Groq docs list these model IDs, so we keep a strong default
# but still allow easy replacement later if needed.
MODEL_NAME = "llama-3.3-70b-versatile"

print("Groq client ready." if client else "Groq API key not found yet. Add GROQ_API_KEY in Colab secrets.")

In [ ]:
# ============================================================
# SMALL HELPER FUNCTIONS
# ============================================================

def extract_json_block(text: str) -> Dict[str, Any]:
    """
    Tries to extract JSON from the model output.
    We keep this helper because LLMs sometimes wrap JSON in markdown fences.
    """
    if not text:
        return {}

    # Remove markdown fences if present.
    cleaned = text.strip()
    cleaned = cleaned.replace("```json", "").replace("```", "").strip()

    # Direct JSON parse first.
    try:
        return json.loads(cleaned)
    except Exception:
        pass

    # Fallback: try to locate the first JSON object in the text.
    match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            return {}

    return {}


def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.3) -> str:
    """
    Safe wrapper over Groq chat completion.
    This function keeps the API call in one place so maintenance is easier.
    """
    if client is None:
        raise ValueError(
            "Groq client is not initialized. Please add GROQ_API_KEY in Colab secrets or environment variables."
        )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content


def clean_list_text(items: List[str]) -> List[str]:
    """
    Utility to normalize list items by removing extra spaces and blank strings.
    """
    return [item.strip() for item in items if str(item).strip()]

In [ ]:
# ============================================================
# DOMAIN KNOWLEDGE / RESOURCE CATALOG
# ============================================================
# This local catalog gives the notebook a deterministic support layer.
# So even if the LLM writes nicely, we still have some structured logic
# behind the recommendations.

RESOURCE_CATALOG = {
    "dsa": [
        "Arrays, strings, hashing, two pointers",
        "Recursion, backtracking, binary search",
        "Stack, queue, linked list, trees, graphs",
        "Dynamic programming and greedy patterns",
        "Weekly timed mock practice"
    ],
    "web development": [
        "HTML, CSS, JavaScript fundamentals",
        "React component design and state management",
        "Node.js + Express backend APIs",
        "MongoDB schema design and queries",
        "Deployment on Vercel / Render"
    ],
    "ai/ml": [
        "Python + NumPy + pandas basics",
        "Data cleaning and feature engineering",
        "Supervised ML with scikit-learn",
        "Model evaluation and validation strategy",
        "Mini end-to-end project deployment"
    ],
    "cloud": [
        "Linux, networking, and HTTP basics",
        "AWS core services overview",
        "IAM, storage, compute, monitoring",
        "Docker basics and container workflows",
        "CI/CD fundamentals"
    ],
    "general": [
        "Communication and structured problem solving",
        "Resume and portfolio refinement",
        "Project documentation",
        "Interview storytelling",
        "Weekly review + KPI tracking"
    ],
}


def heuristic_domain_detect(goal: str) -> str:
    """
    Simple deterministic classifier.
    We keep this because a hybrid system is stronger than depending only on one prompt.
    """
    goal_lower = goal.lower()

    if any(word in goal_lower for word in ["dsa", "leetcode", "coding interview", "problem solving"]):
        return "dsa"
    if any(word in goal_lower for word in ["react", "mern", "frontend", "backend", "web"]):
        return "web development"
    if any(word in goal_lower for word in ["machine learning", "data science", "deep learning", "ai"]):
        return "ai/ml"
    if any(word in goal_lower for word in ["aws", "cloud", "devops", "docker"]):
        return "cloud"
    return "general"


def skill_gap_scan(goal: str, current_skills: str) -> Dict[str, Any]:
    """
    Compares the user's current skills with a target domain resource list.
    This is a lightweight explainable layer.
    """
    domain = heuristic_domain_detect(goal)
    recommended_topics = RESOURCE_CATALOG.get(domain, RESOURCE_CATALOG["general"])

    current_lower = current_skills.lower()
    missing_topics = [topic for topic in recommended_topics if topic.lower() not in current_lower]

    return {
        "detected_domain": domain,
        "recommended_topics": recommended_topics,
        "missing_topics": missing_topics[:5],
    }

In [ ]:
# ============================================================
# MULTI-STEP ADVISOR PIPELINE
# ============================================================

def stage_1_analyze_user(goal: str, current_skills: str, time_horizon_weeks: int, constraints: str) -> Dict[str, Any]:
    """
    Stage 1:
    Understand the user context and convert raw text into a structured view.
    """
    heuristic_view = skill_gap_scan(goal, current_skills)

    system_prompt = (
        "You are a strategic advisor. "
        "Return only valid JSON with keys: goal_summary, user_level, priority_focus, "
        "main_risks, recommended_style."
    )

    user_prompt = f"""
    Goal: {goal}
    Current skills: {current_skills}
    Time horizon (weeks): {time_horizon_weeks}
    Constraints: {constraints}

    Analyze this user and return short, practical JSON only.
    """

    llm_text = call_llm(system_prompt, user_prompt)
    llm_json = extract_json_block(llm_text)

    return {
        "goal": goal,
        "current_skills": current_skills,
        "time_horizon_weeks": time_horizon_weeks,
        "constraints": constraints,
        "heuristic_view": heuristic_view,
        "llm_analysis": llm_json,
    }


def stage_2_generate_options(stage1_output: Dict[str, Any]) -> Dict[str, Any]:
    """
    Stage 2:
    Generate multiple possible strategies.
    We ask for 3 options so the system behaves like an advisor, not a one-line chatbot.
    """
    system_prompt = (
        "You are a planning expert. Return only valid JSON with key 'options'. "
        "Each option must include: name, why_it_works, tradeoff, weekly_focus."
    )

    user_prompt = f"""
    User goal: {stage1_output['goal']}
    Current skills: {stage1_output['current_skills']}
    Time horizon: {stage1_output['time_horizon_weeks']} weeks
    Constraints: {stage1_output['constraints']}
    Heuristic gaps: {stage1_output['heuristic_view']['missing_topics']}

    Generate 3 realistic strategy options.
    """

    llm_text = call_llm(system_prompt, user_prompt, temperature=0.4)
    llm_json = extract_json_block(llm_text)

    return {
        "options": llm_json.get("options", []),
        "raw_text": llm_text,
    }


def stage_3_prioritize_plan(stage1_output: Dict[str, Any], stage2_output: Dict[str, Any]) -> Dict[str, Any]:
    """
    Stage 3:
    Select the best strategy and convert it into an execution plan.
    """
    system_prompt = (
        "You are an execution-focused roadmap architect. Return only valid JSON with keys: "
        "best_option_name, reason_for_selection, action_plan_30_60_90, weekly_kpis, risk_mitigation."
    )

    user_prompt = f"""
    Goal: {stage1_output['goal']}
    Current skills: {stage1_output['current_skills']}
    Time horizon: {stage1_output['time_horizon_weeks']} weeks
    Constraints: {stage1_output['constraints']}
    Strategy options: {json.dumps(stage2_output['options'], indent=2)}

    Choose the strongest option and create a practical 30/60/90 style action plan.
    """

    llm_text = call_llm(system_prompt, user_prompt, temperature=0.2)
    llm_json = extract_json_block(llm_text)

    return llm_json if llm_json else {"raw_text": llm_text}


def stage_4_final_response(stage1_output: Dict[str, Any], stage2_output: Dict[str, Any], stage3_output: Dict[str, Any]) -> Dict[str, Any]:
    """
    Stage 4:
    Convert structured plan into a human-friendly advisor response.
    """
    system_prompt = (
        "You are a senior advisor. Write a final answer in clear English with sections: "
        "Summary, Best Path, Execution Plan, Risks, Final Recommendation."
    )

    user_prompt = f"""
    Stage 1 analysis:
    {json.dumps(stage1_output, indent=2)}

    Stage 2 options:
    {json.dumps(stage2_output, indent=2)}

    Stage 3 selected plan:
    {json.dumps(stage3_output, indent=2)}

    Write a polished final advisory response.
    """

    final_text = call_llm(system_prompt, user_prompt, temperature=0.3)

    return {
        "final_response": final_text
    }


def run_advisor_pipeline(goal: str, current_skills: str, time_horizon_weeks: int, constraints: str) -> Dict[str, Any]:
    """
    Main orchestrator.
    This is the single place that runs the full advisor workflow end-to-end.
    """
    stage1 = stage_1_analyze_user(goal, current_skills, time_horizon_weeks, constraints)
    stage2 = stage_2_generate_options(stage1)
    stage3 = stage_3_prioritize_plan(stage1, stage2)
    stage4 = stage_4_final_response(stage1, stage2, stage3)

    return {
        "stage_1_analysis": stage1,
        "stage_2_options": stage2,
        "stage_3_selected_plan": stage3,
        "stage_4_final_output": stage4,
    }

In [ ]:
# ============================================================
# FASTAPI BACKEND
# ============================================================
# This makes the notebook more production-style.
# Instead of directly calling only Python functions, we expose logic as APIs.

app = FastAPI(title="Multi-Step Intelligent Advisor System API")


class AdvisorRequest(BaseModel):
    goal: str = Field(..., description="What the user wants to achieve")
    current_skills: str = Field(..., description="Current knowledge / tools / experience")
    time_horizon_weeks: int = Field(..., ge=1, le=52, description="Number of weeks available")
    constraints: str = Field(default="", description="Time, budget, confidence, background or other limits")


@app.get("/health")
def health_check():
    return {"status": "ok", "message": "Advisor API is running"}


@app.post("/analyze")
def analyze_only(payload: AdvisorRequest):
    return stage_1_analyze_user(
        goal=payload.goal,
        current_skills=payload.current_skills,
        time_horizon_weeks=payload.time_horizon_weeks,
        constraints=payload.constraints,
    )


@app.post("/advise")
def advise(payload: AdvisorRequest):
    return run_advisor_pipeline(
        goal=payload.goal,
        current_skills=payload.current_skills,
        time_horizon_weeks=payload.time_horizon_weeks,
        constraints=payload.constraints,
    )

In [ ]:
# ============================================================
# API TESTING INSIDE COLAB USING TESTCLIENT
# ============================================================
# This is useful because in Colab we do not always want to run a full public server.
# TestClient lets us validate FastAPI endpoints directly in the notebook.

test_client = TestClient(app)

sample_payload = {
    "goal": "I want to become interview-ready for a Java backend developer role",
    "current_skills": "Java, OOP, basic SQL, basic DSA, little bit of Spring Boot",
    "time_horizon_weeks": 12,
    "constraints": "I can give 2 to 3 hours on weekdays and more on weekends"
}

# Uncomment to test after adding your GROQ_API_KEY.
# response = test_client.post("/advise", json=sample_payload)
# print(response.status_code)
# print(response.json()["stage_4_final_output"]["final_response"][:1500])

In [ ]:
# ============================================================
# GRADIO UI
# ============================================================
# This gives a clean Colab-friendly interface.
# The UI calls the same backend logic, so demo + architecture stay aligned.

def advisor_ui(goal, current_skills, time_horizon_weeks, constraints):
    try:
        result = run_advisor_pipeline(
            goal=goal,
            current_skills=current_skills,
            time_horizon_weeks=int(time_horizon_weeks),
            constraints=constraints,
        )

        # Pull clean text blocks for the UI.
        stage1 = json.dumps(result["stage_1_analysis"]["llm_analysis"], indent=2)
        stage3 = json.dumps(result["stage_3_selected_plan"], indent=2)
        final_answer = result["stage_4_final_output"]["final_response"]

        return stage1, stage3, final_answer

    except Exception as e:
        return f"Error: {e}", "", ""


with gr.Blocks() as demo:
    gr.Markdown("# Multi-Step Intelligent Advisor System")
    gr.Markdown("Enter a goal, your current skills, and your constraints. The system will analyze, plan, and advise step by step.")

    with gr.Row():
        goal = gr.Textbox(label="Goal", placeholder="Example: I want to become job-ready in data science")
        current_skills = gr.Textbox(label="Current Skills", lines=4, placeholder="Example: Python, pandas, SQL basics, statistics basics")
    with gr.Row():
        time_horizon = gr.Slider(1, 52, value=12, step=1, label="Time Horizon (Weeks)")
        constraints = gr.Textbox(label="Constraints", lines=3, placeholder="Example: College schedule, low confidence in ML math, limited daily time")

    run_btn = gr.Button("Generate Advice")

    stage1_box = gr.Textbox(label="Stage 1 - Structured Analysis", lines=12)
    stage3_box = gr.Textbox(label="Stage 3 - Selected Plan", lines=14)
    final_box = gr.Textbox(label="Final Advisor Output", lines=18)

    run_btn.click(
        fn=advisor_ui,
        inputs=[goal, current_skills, time_horizon, constraints],
        outputs=[stage1_box, stage3_box, final_box]
    )

# Uncomment this in Colab when you want the interactive app.
# demo.launch(share=True)

## Final Notes
This notebook is strong for:
- major project demos
- architecture explanation
- API + UI combined submission
- showing **multi-step orchestration** instead of single prompt dependency

To run fully:
1. Add `GROQ_API_KEY` in Colab Secrets  
2. Run all cells  
3. Uncomment the test cell or `demo.launch(share=True)`